<a href="https://colab.research.google.com/github/jothikah/jothikah-OS-Lab-expr/blob/main/OS_Expr6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile semaphore.c
#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>
#include <sys/wait.h>
#include <semaphore.h>

int main() {
    sem_t semaphore;

    // Initialize semaphore
    // 0 = shared between processes
    // 1 = initial value
    sem_init(&semaphore, 0, 1);

    pid_t pid = fork();

    if (pid < 0) {
        perror("Fork failed");
        exit(1);
    }

    if (pid == 0) {
        // Child process
        printf("Child process waiting for semaphore...\n");

        // Wait operation (P)
        sem_wait(&semaphore);

        printf("Child entered critical section.\n");
        sleep(2);
        printf("Child leaving critical section.\n");

        // Signal operation (V)
        sem_post(&semaphore);

        exit(0);
    }
    else {
        // Parent process
        printf("Parent process waiting for semaphore...\n");

        // Wait operation (P)
        sem_wait(&semaphore);

        printf("Parent entered critical section.\n");
        sleep(2);
        printf("Parent leaving critical section.\n");

        // Signal operation (V)
        sem_post(&semaphore);

        wait(NULL);

        // Destroy semaphore
        sem_destroy(&semaphore);

        printf("Semaphore destroyed.\n");
    }

    return 0;
}

Writing semaphore.c


In [2]:
!gcc semaphore.c -o semaphore -pthread
!./semaphore

Parent process waiting for semaphore...
Parent entered critical section.
Child process waiting for semaphore...
Child entered critical section.
Child leaving critical section.
Parent leaving critical section.
Semaphore destroyed.


In [3]:
%%writefile semaphore.sh
#!/bin/bash

LOCKFILE="/tmp/my_semaphore.lock"

# Wait operation
wait_semaphore() {
    while [ -e "$LOCKFILE" ]
    do
        sleep 1
    done

    touch "$LOCKFILE"
}

# Signal operation
signal_semaphore() {
    rm -f "$LOCKFILE"
}

echo "Semaphore Implementation"
echo "------------------------"

echo "Process 1 waiting for semaphore..."
wait_semaphore

echo "Process 1 entered critical section."
sleep 2
echo "Process 1 leaving critical section."

signal_semaphore

echo ""
echo "Process 2 waiting for semaphore..."
wait_semaphore

echo "Process 2 entered critical section."
sleep 2
echo "Process 2 leaving critical section."

signal_semaphore

echo ""
echo "Semaphore demonstration completed."

Writing semaphore.sh


In [4]:
!chmod +x semaphore.sh
!./semaphore.sh

Semaphore Implementation
------------------------
Process 1 waiting for semaphore...
Process 1 entered critical section.
Process 1 leaving critical section.

Process 2 waiting for semaphore...
Process 2 entered critical section.
Process 2 leaving critical section.

Semaphore demonstration completed.
